### Udaplay_02_solution_project — load, process, and format the provided game JSON files.

This solution 
1. loads, processes, and formats the games.json file
2. creates an agent with necessary tools
3. makes some queries
4. displays the result, how the agent's thinking process was

## 1 Setting up dependencies

import the necessary packages and libraries and load environment

In [1]:
from dotenv import load_dotenv
import os

from lib.agents import Agent
from lib.messages import BaseMessage
from typing import List

load_dotenv("config.env")
assert os.getenv("OPENAI_API_KEY") is not None
assert os.getenv("TAVILY_API_KEY") is not None

# 1.1 Create printer function

implement message printer private function to better visualize thinking process and output of the agent

In [2]:
def _print_messages(messages: List[BaseMessage]):
    for m in messages:
        print(
            f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})"
        )

# 2 RAG, Tavily Client, and Long Term Memory initialization


In [3]:
import json
from lib.tooling import tool
from tavily import TavilyClient
from chromadb.api.types import QueryResult
from typing import Dict
from datetime import datetime
from lib.llm import LLM
from lib.rag import RAG
import os
from lib.vector_db import VectorStoreManager
from lib.documents import Document,Corpus
from lib.tooling import Tool
from lib.memory import LongTermMemory, MemoryFragment
from lib.agents import Agent
from lib.evaluation import EvaluationReport
from lib.parsers import PydanticOutputParser


tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

rag_llm = LLM(model="gpt-4o-mini", temperature=0.3)
CHROMA_PERSIST_DIR = os.path.join(os.path.abspath("."), "chroma_db")
vector_store_manager = VectorStoreManager(os.getenv("OPENAI_API_KEY"), persist_directory=CHROMA_PERSIST_DIR)
long_term_memory = LongTermMemory(vector_store_manager)

def _load_game_documents():
    games_dir = os.path.join(os.path.abspath("."), "games")
    documents = []
    for filename in sorted(os.listdir(games_dir)):
        if filename.endswith(".json"):
            with open(os.path.join(games_dir, filename), "r", encoding="utf-8") as f:
                g = json.load(f)
            documents.append(
                Document(
                    content=(
                        f"{g['Name']} is available on {g['Platform']}. "
                        f"Genre: {g.get('Genre', 'Unknown')}. "
                        f"Publisher: {g.get('Publisher', 'Unknown')}. "
                        f"Year of release: {g.get('YearOfRelease', 'Unknown')}. "
                        f"{g.get('Description', '')}"
                    ),
                    metadata={
                        "game": g["Name"],
                        "platform": g["Platform"],
                        "genre": g.get("Genre", ""),
                        "publisher": g.get("Publisher", ""),
                        "year": g.get("YearOfRelease", ""),
                    },
                )
            )
    return Corpus(documents)


long_term_memory.vector_store.add(_load_game_documents())
print(f"Loaded {len(_load_game_documents())} game documents into the vector store.")

games_rag = RAG(llm=rag_llm, vector_store=long_term_memory.vector_store)

Loaded 15 game documents into the vector store.


# 3 Define tools to be used by the agent

In [4]:
@tool
def retrieve_game(game_name:str) -> QueryResult:
    """
    Retrieves game information from the vector store based on the game name.
    args:
        game_name (str): The name of the game to retrieve
    returns:
        QueryResult: The result of the query
    """
    return games_rag.invoke(game_name).get_final_state()["answer"]

@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
        question (str): original question from user
        retrieved_docs (str): retrieved documents most similar to the user query in the Vector Database
    returns:
        dict with:
        - useful: whether the documents are useful to answer the question
        - description: description about the evaluation result
    """
    judge_llm = LLM(model="gpt-4o-mini", temperature=0.0)
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Query: {question}\n\n"
        f"Retrieved Documents:\n{retrieved_docs}"
    )
    response = judge_llm.invoke(input=prompt, response_format=EvaluationReport)
    parser = PydanticOutputParser(model_class=EvaluationReport)
    result = parser.parse(response)
    return {"useful": result.useful, "description": result.description}

@tool
def game_web_search(query: str, search_depth: str = "advanced") -> Dict:
    """
    Search the web using Tavily API
    args:
        query (str): Search query
        search_depth (str): Type of search - 'basic' or 'advanced' (default: advanced)
    """

    # Perform the search
    search_result = tavily_client.search(
        query=query,
        search_depth=search_depth,
        include_answer=True,
        include_raw_content=False,
        include_images=False,
    )

    # Format the results
    formatted_results = {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "citations": [
            {"title": r.get("title", ""), "url": r.get("url", "")}
            for r in search_result.get("results", [])
        ],
        "search_metadata": {"timestamp": datetime.now().isoformat(), "query": query},
    }

    return formatted_results

def build_memory_registration_tool(ltm:LongTermMemory, owner:str, namespace:str):
    """
    Create a tool for agents to register new memories.
    
    This factory function creates a tool that allows AI agents to store new
    information about users in the long-term memory system. The tool is
    pre-configured with specific owner and namespace parameters.
    
    Args:
        ltm (LongTermMemory): The memory system instance to use
        owner (str): User identifier for memory ownership
        namespace (str): Namespace for organizing memories
        
    Returns:
        Tool: A configured tool for memory registration
    """
    def _register(content:str):
        ltm.register(
            MemoryFragment(
                content=content, 
                owner=owner,
                namespace=namespace
            )
        )
        return "Saved new memory"

    return Tool(
        func=_register, 
        name="register_memory", 
        description=(
            "Register a new memory about the information of the game for future reference. "
            "Args:\n"
            "    content: The information to save"
        )
    )

def build_memory_search_tool(ltm:LongTermMemory, owner:str, namespace:str):
    """
    Create a tool for agents to search existing memories.
    
    This factory function creates a tool that allows AI agents to retrieve
    relevant memories from the long-term memory system based on semantic
    similarity to a search query.
    
    Args:
        ltm (LongTermMemory): The memory system instance to use
        owner (str): User identifier for memory ownership
        namespace (str): Namespace to search within
        
    Returns:
        Tool: A configured tool for memory search
    """
    def _search(content:str):
        result = ltm.search(
            query_text=content,
            owner=owner,
            namespace=namespace,
            limit=3,
        )
        return str(tuple(zip(result.fragments, result.metadata['distances'])))

    return Tool(
        func=_search, 
        name="search_memory", 
        description=(
            "Search for a stored memory or preference about the game, " 
            "so it's useful as a context.\n"
            "Args:\n"
            "    content: The information to look for"
        )
    )

# 4 Define a simple agent

Define the agent with necessary tools to look through the RAG first, then evaluate the found answer, if the answer is not satisfactory fall back to long term memory first, if still not found, fall back to web search tool. There are also tools to look through the long term memory and save a new memory to the long term memory

In [5]:
my_agent = Agent(
    model_name="gpt-4o-mini",
    tools=[
        retrieve_game,
        evaluate_retrieval,
        build_memory_search_tool(long_term_memory, "omer_oksuzler", "games"),
        game_web_search,
        build_memory_registration_tool(long_term_memory, "omer_oksuzler", "games"),
    ],
    temperature=0.3,
    instructions=(
        "You are a helpful assistant for game recommendations and information retrieval. "
        "When answering a query about a game, you MUST follow this exact tool-use order:\n"
        "1. Call retrieve_game first to search the local game database.\n"
        "1.1 if a game is found, call evaluate_retrieval to check if the retrieved game information is relevant and sufficient to answer the user's query. If evaluate_retrieval returns useful you can persist the memory and finish\n"
        "2. Only if retrieve_game returns no result or insufficient information, call the long-term memory search tool to check previously stored knowledge.\n"
        "3. Only if the long-term memory search also returns no useful result, call game_web_search to find the information online.\n"
        "4. if you used game_web_search, you MUST put the necessary proper citation of the source in the final answer.\n"
        "5. After using game_web_search, you MUST always call the long-term memory registration tool to persist the retrieved information for future use. And Add citations to every final answer that uses web search, not just to the raw tool output.\n"
        "Never skip steps or jump ahead. Always incorporate all retrieved information into your final response. Be concise and informative."
    ),
)

# 5 Invoke the agent and print results

Create runs that invoke the agent and prints the result, thinking process, and tool usage

In [6]:
run1 = my_agent.invoke(
    query="When Pokémon Gold and Silver was released?", session_id="session_1"
)
run2 = my_agent.invoke(
    query="Which one was the first 3D platformer Mario game?", session_id="session_2"
)

run3 = my_agent.invoke(
    query="Was Halo Infinite realeased for Playstation 5?", session_id="session_1"
)

run4 = my_agent.invoke(
    query="Was Mortal Kombat X realeased for Playstation 5?", session_id="session_2"
)

print("\nMessages from run 1:")
messages = run1.get_final_state()["messages"]
_print_messages(messages)

print("\nMessages from run 2:")
messages = run2.get_final_state()["messages"]
_print_messages(messages)

print("\nMessages from run 3:")
messages = run3.get_final_state()["messages"]
_print_messages(messages)

print("\nMessages from run 4:")
messages = run4.get_final_state()["messages"]
_print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_execut

# 6 Run Report

Summary of each query: tools invoked, final answer, and information source.


In [7]:
from lib.messages import AIMessage, UserMessage, ToolMessage

_SOURCE_PRIORITY = [
    ("game_web_search", "Web search (Tavily)"),
    ("search_memory", "Long-term memory"),
    ("retrieve_game", "Local RAG"),
]


def _build_report(label: str, run) -> None:
    messages = run.get_final_state()["messages"]

    # Query — first user message
    query = next((m.content for m in messages if isinstance(m, UserMessage)), "N/A")

    # Tools used — deduplicated, in call order, via ToolMessages
    tools_used = []
    seen = set()
    for m in messages:
        if isinstance(m, ToolMessage) and m.name not in seen:
            tools_used.append(m.name)
            seen.add(m.name)

    # Final answer — last non-empty AIMessage
    final_answer = next(
        (
            m.content
            for m in reversed(messages)
            if isinstance(m, AIMessage) and m.content
        ),
        "N/A",
    )

    # Source — highest-priority tool that was actually called
    source = "N/A"
    for tool_name, label_str in _SOURCE_PRIORITY:
        if tool_name in seen:
            source = label_str
            break

    print(f"{'─' * 60}")
    print(f"  {label}")
    print(f"{'─' * 60}")
    print(f"  Query   : {query}")
    print(f"  Tools   : {', '.join(tools_used) if tools_used else 'none'}")
    print(f"  Source  : {source}")
    print(f"  Answer  : {final_answer}")
    print()


for label, run in [
    ("Run 1", run1),
    ("Run 2", run2),
    ("Run 3", run3),
    ("Run 4", run4),
]:
    _build_report(label, run)

────────────────────────────────────────────────────────────
  Run 1
────────────────────────────────────────────────────────────
  Query   : When Pokémon Gold and Silver was released?
  Tools   : retrieve_game, evaluate_retrieval
  Source  : Local RAG
  Answer  : Pokémon Gold and Silver was released in 1999 for the Game Boy Color. These games are part of the second generation of Pokémon titles, introducing new regions, Pokémon, and gameplay mechanics.

────────────────────────────────────────────────────────────
  Run 2
────────────────────────────────────────────────────────────
  Query   : Which one was the first 3D platformer Mario game?
  Tools   : retrieve_game, evaluate_retrieval
  Source  : Local RAG
  Answer  : The first 3D platformer Mario game is **Super Mario 64**, which was released in 1996 for the Nintendo 64. This game marked a significant milestone in gaming, introducing players to a fully 3D environment where Mario's quest is to rescue Princess Peach.

────────────────